# Gallbladder Cancer Dynamic Survival Nomogram
Deploy the Shiny for Python application from GitHub in Google Colab. Raw SEER case-level data are not required and must not be uploaded.

In [ ]:
REPO_URL = 'https://github.com/xkfy1979/gallbladder-survival-nomogram.git'
APP_DIR = '/content/gallbladder-survival-nomogram'

In [ ]:
import os, subprocess, shutil
if os.path.exists(APP_DIR): shutil.rmtree(APP_DIR)
subprocess.run(['git','clone','--depth','1',REPO_URL,APP_DIR],check=True)
subprocess.run(['pip','install','-q','-r',f'{APP_DIR}/requirements.txt'],check=True)

In [ ]:
import urllib.request, time, re
cloudflared='/content/cloudflared'
urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',cloudflared)
os.chmod(cloudflared,0o755)
app_log=open('/content/shiny.log','w'); tunnel_log=open('/content/tunnel.log','w')
app_proc=subprocess.Popen(['shiny','run','--host','0.0.0.0','--port','8000','app.py'],cwd=APP_DIR,stdout=app_log,stderr=subprocess.STDOUT)
time.sleep(8)
tunnel_proc=subprocess.Popen([cloudflared,'tunnel','--url','http://127.0.0.1:8000','--no-autoupdate'],stdout=tunnel_log,stderr=subprocess.STDOUT)
time.sleep(10)
text=open('/content/tunnel.log').read(); urls=re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com',text)
print('Public application URL:', urls[-1] if urls else 'URL not ready; rerun this cell after a few seconds.')